 # Talking Heroes #
 Three Dungeon'n'Dragons Characters created in week 1 excersise take a talk in front of a mysteriously looking cave.<br>
 <i>Excersise based on Week 2 info (incl. LiteLLM)</i>


In [1]:
# base imports, settings and initialization of variables 

import json
import os

from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from litellm import completion
from openai import OpenAI

# load environment variables
load_dotenv(override=True)
MODELS = ["openai/gpt-5.4", "anthropic/claude-sonnet-4-6", "gemini/gemini-2.5-flash"]

# assign API keys from env and check if key exists
error_msg = []
if not (openai_api_key := os.getenv('OPENAI_API_KEY')) or not openai_api_key.startswith("sk-proj-"):
    error_msg.append("No OPEN API key was found or is invalid (doesn't start sk-proj-)")
if not (anthropic_api_key := os.getenv('ANTHROPIC_API_KEY')) or not anthropic_api_key.startswith("sk-ant-"):
    error_msg.append("No ANTHROPIC API key was found or is invalid (doesn't start sk-ant-")
if not (google_api_key := os.getenv('GOOGLE_API_KEY')):
    error_msg.append("No GOOGLE API key was found")
if error_msg:
    raise("\n".join(error_msg))
print("API keys found and look good so far!")

# openai = OpenAI()


API keys found and look good so far!


In [2]:
# read characters from text file
with open(os.path.join(os.getcwd(), "..", "data", "dnd_characters.txt"), "r", encoding="utf-8") as datafile:
    content = datafile.read()

# split content into 3 characters with AI :)
user_prompt = """You are provided with a text containing 3 different game characters. 
Characters are separated by text Character N:, where N is an ordinary number. Your task is to extract each character, 
pick name and a summary of each character describing personality and communication style with maximum 333 characters.
Output format must be JSON and only JSON with structure: 
[
    {
        "name": "name of character N",
        "description": "summary of character N"
    }
] 
If there would be more than 3 characters then take the first three only. 
If there would be less than 3 characters then return JSON and only JSON with structure:
[
    {
        "error": "Error in content - not enough characters"
    }
]
The text containing 3 characters is: """ + content

response = completion(
    model="openai/gpt-5-mini", 
    messages=[{'role': 'user', 'content': user_prompt}], 
    reasoning_effort="minimal"
    )
character_list = json.loads(response.choices[0].message.content)

if err := character_list[0].get('error', False):
    raise(character_list[0]['error'])



In [3]:
[(item['name'], item['description']) for item in character_list]

[('Edron "Don" Gearwhistle',
  'Pragmatic, curious rock gnome artificer and tech-entrepreneur. Nerdy, detail-focused explainers who loves tinkering and teaching; persuasive presenter with dry snark. Communicates analytically, gives long explanations, prioritizes progress and practical tools.'),
 ('Queen Elsbeth Alexandra of Balmoral',
  'Dignified protector aasimar paladin and monarchly steward. Measured, ceremonious, duty-first communicator who calms councils and follows protocol. Polite, composed, privately strong; favors law, ceremony, and stability over emotional displays.'),
 ('Arnold von Thal',
  'Goliath paladin, former strongman turned governor. Direct, confident, showman with rugged pragmatism; enjoys applause and commands presence. Speaks plainly and forcefully, driven by duty and discipline but prone to bluntness and overconfidence.')]

In [4]:
# 1. create system prompt (universal for all 3 agents) and user prompts
# 2. run 9 rounds of conversation and show tokens/cents spending with LiteLLM 

import litellm
import logging

from litellm.caching.caching import CacheMode

litellm._turn_on_debug()
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

system_prompt = """ 
    Act as character from famous table-top role playing game Dungeon and Dragons. You are standing in front of a mysteriosly 
    looking cave, hearing some noise from inside, but due to darkness you cannot see what is there. 
    Your task is to chat with your teammates according to role details you get in user prompt and decide what to do.
    Dialogues are prefixed with character Name to make it clear who says what. Characters may react each other 
    marking partner with naming, for example "Alex, I totally agree with you but there is some issue...".
    Response must be shorter than 333 characters.
"""

spending = {}

user_prompts = [
 f"Your name is {character_list[0]['name']}, you act as character with following description: {character_list[0]['description']}. Your task is to persuade your teammates to find any other way into the cave, because you FEEL the main entrace you are standing in front of is a trap. You are a snearky speaker who loves jokes.",
 f"Your name is {character_list[1]['name']}, you act as character with following description: {character_list[1]['description']}. Your task is to persuade your teammates to leave the cave, go to the nearest village and find someone you could provide a help, because it may improve your karma better than to attack some creatures and get their treasures. You are wise but pragmatic speaker.",
 f"Your name is {character_list[2]['name']}, you act as character with following description: {character_list[2]['description']}. Goal you want the most would be to run directly inside the cave, attack anything inside and be again the hero with the most powerful muscles and shiny smile. You commonly don't care about someone elses feelings or talks, but you know you are team of three and must get to a democratic agreement."
]

chat_history = "\nChat history:\n"
for i in range(9):
    speaker_coordinate = i%3 
    response = completion(
        model=MODELS[speaker_coordinate], 
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompts[speaker_coordinate] + chat_history}]
    )
    chat_history += f"{response.choices[0].message.content}\n\n"

    if not spending.get(MODELS[speaker_coordinate]):
        spending[MODELS[speaker_coordinate]] = {
            "total_tokens": response.usage.total_tokens,
            "total_cost": response._hidden_params["response_cost"]*100
        }
    else:
        spending[MODELS[speaker_coordinate]]["total_tokens"] += response.usage.total_tokens
        spending[MODELS[speaker_coordinate]]["total_cost"] += response._hidden_params["response_cost"]*100

        
display(Markdown(chat_history))

print("\nChat spendings:\n", json.dumps(spending, indent=4))



Chat history:
Edron “Don” Gearwhistle: Team, call it intuition wrapped in engineering: this entrance screams “fatal design feature.” Hear that echo? Bad geometry, likely a kill-box. Let’s find a vent, fissure, side shaft—anything less murdery. I prefer innovation over becoming cave décor.

Queen Elsbeth Alexandra: Don raises valid structural concerns. Yet I propose we forgo this cave entirely. Villagers nearby surely need aid far more than we need whatever lurks within. Serving the living yields greater honor than plundering shadows. Shall we proceed with purpose rather than peril?

Arnold von Thal: Don, a good shield and strong arm make short work of any "kill-box." Elsbeth, our duty stands here, facing this darkness. The noise demands a hero, not a retreat. My vote is to charge in, claim victory, and move on! Are we agreed?

Edron “Don” Gearwhistle: Arnold, “charge in” is a fine plan if your goal is testing gravity with your face. Elsbeth, skipping entirely wastes intel. Noise means occupants; occupants need exits. Let’s circle, map airflow, find a service entrance. Same mission, fewer funeral expenses.

Queen Elsbeth Alexandra: Arnold, valor without purpose is merely noise matching noise. Don, your caution is prudent, yet mapping this cave serves ambition, not duty. The village represents *certain* need. Real people, real suffering. That outweighs *probable* treasure. I move we depart immediately.

Arnold: Elsbeth, *this* is our duty, not some distant village! And Don, "cowards" find vents, heroes make their own way! My strength, my shield, will clear this "kill-box" in moments. We walk in, we smash, we emerge victorious. Final vote: in or out? Let's not spend all day debating shadows.

Edron “Don” Gearwhistle: Arnold, heroes who ignore obvious traps become instructional diagrams. I’m not voting “out”; I’m voting “smarter in.” Elsbeth, five minutes to scout airflow and side cracks could save lives *and* reveal if villagers are tied to this. Let me find the non-suicidal entrance.

Arnold, Don — hear me. We debate shadows while real souls suffer in yonder village. A paladin's oath binds me to *certain* good, not *probable* glory. Don's caution is wisdom; redirect it toward people who need us. I move we depart. Now. Duty calls clearly.

Arnold: Elsbeth, our duty is *here*, facing this threat. Don seeks a way *in*, as do I. Your compassion is noted, but a paladin's oath also binds us to confront darkness. I say we go in, as heroes should. Don, are you with me? Let's end this debate.




Chat spendings:
 {
    "openai/gpt-5.4": {
        "total_tokens": 1562,
        "total_cost": 0.6505000000000001
    },
    "anthropic/claude-sonnet-4-6": {
        "total_tokens": 1942,
        "total_cost": 0.8333999999999999
    },
    "gemini/gemini-2.5-flash": {
        "total_tokens": 4738,
        "total_cost": 0.7942200000000001
    }
}
